# World Maps — scaled and cross-family signals on the physical-event videos (Colab)

Runs the Pilot 0 protocol (leak-free, same 200 ms bins) on larger models and other families, then the same pre-specified analysis.

| Model | Signal |
|---|---|
| V-JEPA 2 ViT-H, ViT-g (256 px) | latent prediction error, other-scene control, pre/post change |
| VideoMAE-large | pixel reconstruction error, future masked |
| DINOv2-large, DINOv2-giant, SigLIP 2 so400m | frame-pair embedding change |
| Qwen3-VL (8B if the GPU has ≥ 20 GB, else 4B) | P(Yes) to "does a new physical event begin in the last 0.3 s?" and 1 − P(Yes) to "was it predictable?" |

**Runtime:** GPU runtime (Runtime → Change runtime type → GPU). A100/L4: about 30–60 min total. T4: about 1.5–2.5 h (the 4B VLM is used).

**When it finishes:** the last cell downloads `world_maps_scaled_results.zip`. Send that file back. Nothing is pushed to GitHub.

In [ ]:
#@title 1. Configuration
MODELS = ["vjepaH", "vjepag", "vmaeL", "dinoL", "dinog", "siglip2", "vlm"]  #@param {type:"raw"}
INCLUDE_VJEPA_G_384 = False   #@param {type:"boolean"}   # slow; adds the 384 px ViT-g
INCLUDE_DINOV3 = False        #@param {type:"boolean"}   # gated on the Hub; needs HF_TOKEN with access granted
SMOKE_TEST_FIRST = True       #@param {type:"boolean"}   # 1 scene x 2 bins per model before the full run
LIMIT_SCENES = None           #@param {type:"raw"}       # e.g. 4 for a partial run; None = all 32
if INCLUDE_VJEPA_G_384: MODELS.append("vjepag384")
if INCLUDE_DINOV3: MODELS.append("dinov3L")

import os, subprocess, torch
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
VLM_ID = "Qwen/Qwen3-VL-8B-Instruct" if gpu_gb >= 20 else "Qwen/Qwen3-VL-4B-Instruct"
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (switch to a GPU runtime)'}  {gpu_gb:.0f} GB -> VLM {VLM_ID}")
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok: os.environ["HF_TOKEN"] = tok; print("HF_TOKEN loaded from Colab secrets")
except Exception:
    pass
print("models:", MODELS)

In [ ]:
#@title 2. Code and dependencies (about 2 min)
%cd /content
import os
if not os.path.exists("world-maps"):
    !git clone -q https://github.com/physicalreasoning/world-maps.git
%cd /content/world-maps
!git rev-parse --short HEAD
!pip install -q "transformers>=5.0" accelerate pillow opencv-python-headless scikit-learn scipy pandas safetensors
import torch, transformers, torchvision, cv2
print("torch", torch.__version__, "| transformers", transformers.__version__, "| torchvision", torchvision.__version__, "| cv2", cv2.__version__)
os.environ["PYTHONPATH"] = "/content/world-maps/src"
os.makedirs("notebooks", exist_ok=True)

In [ ]:
#@title 3. Public data (about 270 MB from OSF + the lab's analysis repo)
%cd /content/world-maps
!bash scripts/download_data.sh
!ls data data/events | head

In [ ]:
%%writefile /content/world-maps/scripts/extract_event_signals_scaled.py
"""Scaled and cross-family signals on the 32 physical-event base videos (Pilot 0 extension).

Same bins, clips and leak-free protocol as ``extract_event_signals.py`` (bin k <-> probe onset
t_k = 0.2 k s; context = the 12 frames before t_k at 15 fps; target = the 4 frames after t_k), run
on larger models and on other families. One model is loaded at a time; the merged table is
``results/events_features_scaled.csv`` (one row per scene x bin) and model metadata (ids,
parameter counts, device, dtype, seconds) go to ``results/events_models_scaled.json``.

Signals
  vjepa{H,g,g384}_surprise / _mismatch / _change   V-JEPA 2 ViT-H and ViT-g (256 px; g also at 384 px):
                                                   latent prediction error, other-scene control, pre/post change
  vmaeL_surprise                                    VideoMAE-large pixel reconstruction error, future tubelets masked
  dino{L,g}_change                                  DINOv2-large / -giant CLS change, frame t_k-0.2 vs t_k+0.2
  dinov3L_change                                    DINOv3 ViT-L/16 (gated on the Hub; needs HF_TOKEN), same readout
  siglip2_change                                    SigLIP 2 so400m image-embedding change, same frames
  vlm_boundary                                      Qwen3-VL: P("Yes" | "does a new physical event begin in the
                                                   last 0.3 s of this clip?") on the 16-frame clip around t_k
  vlm_unpredictable                                 Qwen3-VL: 1 - P("Yes" | "was the last 0.3 s predictable from
                                                   the earlier frames?")

The VLM readouts are a separate family: a language-grounded judgment of the clip, not a
predictive representation. They are reported next to the video-model signals, not pooled with them.
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import time

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from extract_event_signals import FPS_MODEL, base_video, clip, clip_fwd, frame_at, load_frames, to_tensor  # noqa: E402

from world_maps.paths import LENGTHS, RESULTS  # noqa: E402

LEN = json.load(open(LENGTHS))
MEAN_IMNET = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD_IMNET = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
DEV = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(0)

# name -> (family, hub id, input size)
MODELS = {
    "vjepaH": ("vjepa2", "facebook/vjepa2-vith-fpc64-256", 256),
    "vjepag": ("vjepa2", "facebook/vjepa2-vitg-fpc64-256", 256),
    "vjepag384": ("vjepa2", "facebook/vjepa2-vitg-fpc64-384", 384),
    "vmaeL": ("videomae", "MCG-NJU/videomae-large", 224),
    "dinoL": ("dinov2", "facebook/dinov2-large", 224),
    "dinog": ("dinov2", "facebook/dinov2-giant", 224),
    "dinov3L": ("dinov3", "facebook/dinov3-vitl16-pretrain-lvd1689m", 224),
    "siglip2": ("siglip2", "google/siglip2-so400m-patch16-256", 256),
    "vlm": ("qwen_vl", "Qwen/Qwen3-VL-8B-Instruct", 256),
}
DEFAULT_SET = ["vjepaH", "vjepag", "vmaeL", "dinoL", "dinog", "siglip2", "vlm"]

PROMPT_BOUNDARY = (
    "This is a short clip of a simple physical scene with a few objects. In the final 0.3 seconds of the "
    "clip, does a new physical event begin, for example a collision, an object starting to fall, an object "
    "entering a container, or an object becoming hidden behind another? Answer Yes or No."
)
PROMPT_PREDICTABLE = (
    "This is a short clip of a simple physical scene with a few objects. Was what happened in the final "
    "0.3 seconds of the clip predictable from the earlier frames? Answer Yes or No."
)


def n_params(m) -> int:
    return int(sum(p.numel() for p in m.parameters()))


def load_frames_cached(cache: dict, scene: str, size: int):
    if size not in cache:
        cache[size] = load_frames(base_video(scene), size)
    return cache[size]


# ---------------------------------------------------------------- per-family runners

def run_vjepa(name, hub, size, scenes, bins_of, log):
    from transformers import VJEPA2Model

    m = VJEPA2Model.from_pretrained(hub).to(DEV).eval()
    per_tub = (size // 16) ** 2
    rows = []
    for si, scene in enumerate(scenes):
        t0 = time.time()
        f = load_frames(base_video(scene), size)
        f_other = load_frames(base_video(scenes[(si + 1) % len(scenes)]), size)
        for k in bins_of(scene):
            t = 0.2 * k
            with torch.no_grad():
                ctx = to_tensor(clip(f, t, 12)).unsqueeze(0).to(DEV)
                full = torch.cat([ctx, to_tensor(clip_fwd(f, t, 4)).unsqueeze(0).to(DEV)], 1)
                full_o = torch.cat([to_tensor(clip(f_other, t, 12)), to_tensor(clip_fwd(f_other, t, 4))], 0).unsqueeze(0).to(DEV)
                enc_ctx = m(ctx, skip_predictor=True).last_hidden_state
                enc_full = m(full, skip_predictor=True).last_hidden_state
                enc_o = m(full_o, skip_predictor=True).last_hidden_state
                cm = [torch.arange(6 * per_tub, device=DEV).unsqueeze(0)]
                tm = [torch.arange(6 * per_tub, 8 * per_tub, device=DEV).unsqueeze(0)]
                pred = m.predictor(encoder_hidden_states=enc_ctx, context_mask=cm, target_mask=tm).last_hidden_state
                tgt = F.layer_norm(enc_full[:, 6 * per_tub:], (enc_full.size(-1),))
                tgt_o = F.layer_norm(enc_o[:, 6 * per_tub:], (enc_o.size(-1),))
                pre = enc_full[:, 4 * per_tub: 6 * per_tub].mean(1)
                post = enc_full[:, 6 * per_tub:].mean(1)
                rows.append({"scene": scene, "bin": k,
                             f"{name}_surprise": float((pred - tgt).abs().mean()),
                             f"{name}_mismatch": float((pred - tgt_o).abs().mean()),
                             f"{name}_change": float(1 - F.cosine_similarity(pre, post).item())})
        log(f"  {name} {scene}: {len(bins_of(scene))} bins in {time.time() - t0:.0f}s")
    return rows, n_params(m)


def run_videomae(name, hub, size, scenes, bins_of, log):
    from transformers import VideoMAEForPreTraining

    m = VideoMAEForPreTraining.from_pretrained(hub).to(DEV).eval()
    per_tub = (size // 16) ** 2
    rows = []
    for scene in scenes:
        t0 = time.time()
        f = load_frames(base_video(scene), size)
        for k in bins_of(scene):
            t = 0.2 * k
            with torch.no_grad():
                v = torch.cat([to_tensor(clip(f, t, 12)), to_tensor(clip_fwd(f, t, 4))], 0).unsqueeze(0).to(DEV)
                mask = torch.zeros(1, 8 * per_tub, dtype=torch.bool, device=DEV)
                mask[:, 6 * per_tub:] = True
                rows.append({"scene": scene, "bin": k, f"{name}_surprise": float(m(pixel_values=v, bool_masked_pos=mask).loss)})
        log(f"  {name} {scene}: {len(bins_of(scene))} bins in {time.time() - t0:.0f}s")
    return rows, n_params(m)


def run_image_change(name, family, hub, size, scenes, bins_of, log):
    """Frame-pair embedding change for image models (DINOv2, DINOv3, SigLIP 2), with explicit preprocessing
    (ImageNet mean/std for DINO; 0.5/0.5 for SigLIP) so no image-processor dependency is needed."""

    from transformers import AutoModel

    kw = {"token": os.environ.get("HF_TOKEN")} if os.environ.get("HF_TOKEN") else {}
    m = AutoModel.from_pretrained(hub, **kw).to(DEV).eval()
    if family == "siglip2":
        mean, std = torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1), torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1)
    else:
        mean, std = MEAN_IMNET, STD_IMNET

    def embed(frames_uint8):
        x = np.stack([cv2.resize(f, (size, size), interpolation=cv2.INTER_AREA) for f in frames_uint8])
        px = ((torch.from_numpy(x).permute(0, 3, 1, 2).float() / 255.0 - mean) / std).to(DEV)
        with torch.no_grad():
            if family == "siglip2":
                vision = m.vision_model if hasattr(m, "vision_model") else m
                out = vision(pixel_values=px)
                return out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state.mean(1)
            out = m(pixel_values=px)
            return out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state[:, 0]

    rows = []
    for scene in scenes:
        t0 = time.time()
        f = load_frames(base_video(scene), max(size, 256))
        for k in bins_of(scene):
            t = 0.2 * k
            e = embed([frame_at(f, t - 0.2), frame_at(f, t + 0.2)])
            rows.append({"scene": scene, "bin": k, f"{name}_change": float(1 - F.cosine_similarity(e[:1], e[1:]).item())})
        log(f"  {name} {scene}: {len(bins_of(scene))} bins in {time.time() - t0:.0f}s")
    return rows, n_params(m)


def run_vlm(name, hub, size, scenes, bins_of, log):
    from transformers import AutoModelForImageTextToText, AutoProcessor

    dtype = torch.bfloat16 if DEV != "cpu" else torch.float32
    try:
        m = AutoModelForImageTextToText.from_pretrained(hub, dtype=dtype, device_map={"": DEV}).eval()
    except TypeError:
        m = AutoModelForImageTextToText.from_pretrained(hub, torch_dtype=dtype, device_map={"": DEV}).eval()
    proc = AutoProcessor.from_pretrained(hub)
    tok = proc.tokenizer
    yes_ids = {tok.encode(s, add_special_tokens=False)[0] for s in ("Yes", " Yes", "yes")}
    no_ids = {tok.encode(s, add_special_tokens=False)[0] for s in ("No", " No", "no")}

    def p_yes(frames_uint8, prompt) -> float:
        messages = [{"role": "user", "content": [{"type": "video"}, {"type": "text", "text": prompt}]}]
        text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        try:
            inputs = proc(text=[text], videos=[frames_uint8], return_tensors="pt",
                          video_metadata=[{"fps": FPS_MODEL, "total_num_frames": len(frames_uint8), "duration": len(frames_uint8) / FPS_MODEL}])
        except TypeError:
            inputs = proc(text=[text], videos=[frames_uint8], return_tensors="pt")
        inputs = {k: (v.to(DEV) if hasattr(v, "to") else v) for k, v in inputs.items()}
        with torch.no_grad():
            logits = m(**inputs).logits[0, -1].float()
        ly = torch.logsumexp(logits[list(yes_ids)], 0)
        ln = torch.logsumexp(logits[list(no_ids)], 0)
        return float(torch.softmax(torch.stack([ly, ln]), 0)[0])

    rows = []
    for scene in scenes:
        t0 = time.time()
        f = load_frames(base_video(scene), size)
        for k in bins_of(scene):
            t = 0.2 * k
            frames = np.concatenate([clip(f, t, 12), clip_fwd(f, t, 4)], 0)  # 16 frames, 15 fps, uint8 (T,H,W,3)
            rows.append({"scene": scene, "bin": k,
                         f"{name}_boundary": p_yes(frames, PROMPT_BOUNDARY),
                         f"{name}_unpredictable": 1.0 - p_yes(frames, PROMPT_PREDICTABLE)})
        log(f"  {name} {scene}: {len(bins_of(scene))} bins in {time.time() - t0:.0f}s")
    return rows, n_params(m)


RUNNERS = {"vjepa2": run_vjepa, "videomae": run_videomae, "qwen_vl": run_vlm}


# ---------------------------------------------------------------- driver

def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--models", nargs="*", default=DEFAULT_SET, help=f"subset of {list(MODELS)}")
    ap.add_argument("--vlm-id", default=None, help="override the Qwen3-VL hub id (e.g. Qwen/Qwen3-VL-4B-Instruct)")
    ap.add_argument("--limit", type=int, default=None, help="only the first N scenes (smoke test)")
    ap.add_argument("--max-bins", type=int, default=None, help="only the first N bins per scene (smoke test)")
    ap.add_argument("--out", default=str(RESULTS / "events_features_scaled.csv"))
    ap.add_argument("--meta", default=str(RESULTS / "events_models_scaled.json"))
    args = ap.parse_args(argv)
    if args.vlm_id:
        MODELS["vlm"] = ("qwen_vl", args.vlm_id, 256)
    scenes = sorted(LEN)[: args.limit]

    def bins_of(scene):
        n = int(np.floor(LEN[scene] / 0.2))
        return list(range(1, (min(n, args.max_bins) if args.max_bins else n) + 1))

    merged = pd.read_csv(args.out) if os.path.exists(args.out) else pd.DataFrame(columns=["scene", "bin"])
    meta = json.load(open(args.meta)) if os.path.exists(args.meta) else {}
    log = lambda s: print(s, flush=True)  # noqa: E731
    log(f"device {DEV}; scenes {len(scenes)}; models {args.models}")
    for name in args.models:
        family, hub, size = MODELS[name]
        done_cols = [c for c in merged.columns if c.startswith(name + "_")]
        if done_cols and set(scenes) <= set(merged.dropna(subset=done_cols).scene.unique()):
            log(f"{name}: already in {args.out}, skipping")
            continue
        log(f"{name}: loading {hub}")
        t0 = time.time()
        try:
            if family in RUNNERS:
                rows, npar = RUNNERS[family](name, hub, size, scenes, bins_of, log)
            else:
                rows, npar = run_image_change(name, family, hub, size, scenes, bins_of, log)
        except Exception as e:  # noqa: BLE001  keep going with the other models; record the failure
            log(f"{name}: FAILED: {type(e).__name__}: {e}")
            meta[name] = {"hub": hub, "family": family, "status": f"failed: {type(e).__name__}: {str(e)[:300]}"}
            json.dump(meta, open(args.meta, "w"), indent=1)
            continue
        df = pd.DataFrame(rows)
        merged = merged.drop(columns=[c for c in merged.columns if c.startswith(name + "_")], errors="ignore")
        merged = df if merged.empty else merged.merge(df, on=["scene", "bin"], how="outer")
        merged.sort_values(["scene", "bin"]).to_csv(args.out, index=False)
        meta[name] = {"hub": hub, "family": family, "input_size": size, "params": npar, "device": DEV,
                      "seconds": round(time.time() - t0), "status": "ok", "scenes": len(scenes)}
        json.dump(meta, open(args.meta, "w"), indent=1)
        log(f"{name}: done in {time.time() - t0:.0f}s ({npar / 1e6:.0f}M params)")
        if DEV == "cuda":
            torch.cuda.empty_cache()
    log(f"wrote {args.out} and {args.meta}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/world-maps/scripts/analyze_events_scaled.py
"""Analysis of the scaled and cross-family signals, with the same pre-specified readouts as Pilot 0.

For every model signal in ``results/events_features_scaled.csv`` (plus the Pilot 0 signals from
``results/events_features.csv`` on the same bins, for a like-for-like comparison):

  B. boundary localization: within-scene AUROC for expert-boundary bins, mean over scenes, 95% scene
     bootstrap, and the paired difference against optical-flow acceleration;
  C. held-out human prediction: leave-one-scene-out logistic regression of trial-level detection on the
     low-level base (lab pixel change, pix_diff, flow_speed, flow_accel, t, t^2) with and without the
     signal; gain in held-out log-likelihood per trial (x1000) with scene-bootstrap CI; Holm over the
     new model signals (the Pilot 0 signals keep their original Holm family and are shown for reference).

Also writes a scale table (family, size, parameters, AUROC, gain) so the within-recipe scale curves
(H4, descriptive) can be read directly. Output: ``results/events_results_scaled.json`` and ``.md``.
"""

from __future__ import annotations

import json
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from analyze_events import LOW, boot_mean, boundary_auc, load_trials, loso_ll, scene_boot_diff, zwithin  # noqa: E402

from world_maps.core import holm_adjust  # noqa: E402
from world_maps.paths import RESULTS  # noqa: E402

FAMILY = {"vjepa": "V-JEPA 2", "vmae": "VideoMAE", "dino": "DINOv2", "dinov3": "DINOv3", "siglip2": "SigLIP 2", "vlm": "Qwen3-VL"}
PILOT0 = {"vjepa_surprise": ("V-JEPA 2", "L", 300e6), "vmae_surprise": ("VideoMAE", "base", 87e6), "dino_change": ("DINOv2", "base", 86e6)}


def family_of(sig: str) -> str:
    stem = sig.split("_")[0]
    for k in sorted(FAMILY, key=len, reverse=True):
        if stem.startswith(k):
            return FAMILY[k]
    return stem


def main(scaled_csv: str, pilot_csv: str, meta_path: str) -> None:
    tr, _ = load_trials()
    sc = pd.read_csv(scaled_csv)
    p0 = pd.read_csv(pilot_csv)[["scene", "bin"] + LOW + list(PILOT0)]
    feat = p0.merge(sc, on=["scene", "bin"], how="inner").dropna().reset_index(drop=True)
    meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {}
    new_sigs = [c for c in sc.columns if c not in ("scene", "bin") and not c.endswith("_mismatch")]
    sigs = list(PILOT0) + new_sigs
    res = {"n_bins": int(len(feat)), "n_scenes": int(feat.scene.nunique()), "models": meta}

    # B
    ref = boundary_auc(feat, "flow_accel")
    resB = {}
    for s in sigs + ["flow_accel"]:
        per = boundary_auc(feat, s)
        m, lo, hi = boot_mean(per.values())
        common = [k for k in per if k in ref]
        resB[s] = dict(auc=m, lo=lo, hi=hi, minus_flow_accel=boot_mean([per[k] - ref[k] for k in common]))
    res["B_boundary_auc"] = resB
    # sanity for V-JEPA sizes: real future predicted better than another scene's future
    res["sanity_surprise_lt_mismatch"] = {s: float((feat[s] < feat[s.replace("_surprise", "_mismatch")]).mean())
                                          for s in new_sigs if s.endswith("_surprise") and s.replace("_surprise", "_mismatch") in feat}

    # C
    p = tr[tr.probe].merge(feat, on=["scene", "bin"], how="inner")
    p["t"] = p.bin * 0.2
    p["t2"] = p.t ** 2
    p["lab_pix"] = p.lab_pix.fillna(p.lab_pix.median())
    p = zwithin(p, sigs + LOW + ["lab_pix"])
    p[["t", "t2"]] = (p[["t", "t2"]] - p[["t", "t2"]].mean()) / p[["t", "t2"]].std()
    base = ["lab_pix", "pix_diff", "flow_speed", "flow_accel", "t", "t2"]
    ll0 = loso_ll(p, base)
    resC = {"n_trials": int(len(p)), "n_scenes": int(p.scene.nunique())}
    pv = {}
    for s in sigs:
        ll1 = loso_ll(p, base + [s])
        est, lo, hi, pval = scene_boot_diff(p, ll0, ll1)
        resC[s] = dict(dLL_per_trial_x1000=1000 * est, lo=1000 * lo, hi=1000 * hi, p=pval)
        if s in new_sigs:
            pv[s] = pval
    for s, adj in holm_adjust(pv).items():
        resC[s]["p_holm"] = adj
    res["C_heldout_human"] = resC

    # scale table
    rows = []
    for s in sigs:
        if s in PILOT0:
            fam, size, npar = PILOT0[s]
        else:
            name = s.split("_")[0]
            fam, size = family_of(s), name.replace(name.rstrip("Lg384H"), "") or "so400m"
            npar = meta.get(name, {}).get("params")
        rows.append({"signal": s, "family": fam, "size": size, "params_M": None if npar is None else round(npar / 1e6),
                     "boundary_auc": resB[s]["auc"], "auc_lo": resB[s]["lo"], "auc_hi": resB[s]["hi"],
                     "gain_x1000": resC[s]["dLL_per_trial_x1000"], "gain_lo": resC[s]["lo"], "gain_hi": resC[s]["hi"],
                     "p_holm": resC[s].get("p_holm")})
    table = pd.DataFrame(rows).sort_values(["family", "params_M"], na_position="last")
    res["scale_table"] = table.to_dict("records")
    json.dump(res, open(RESULTS / "events_results_scaled.json", "w"), indent=1)
    md = ["| Signal | Family | Size | Params (M) | Boundary AUROC [95% CI] | Held-out gain ×10⁻³ nats/trial [95% CI] | Holm p |", "|---|---|---|---|---|---|---|"]
    for r in table.to_dict("records"):
        ph = "ref" if r["p_holm"] is None else f"{r['p_holm']:.3f}"
        md.append(f"| {r['signal']} | {r['family']} | {r['size']} | {r['params_M'] or ''} | {r['boundary_auc']:.2f} [{r['auc_lo']:.2f}, {r['auc_hi']:.2f}] | "
                  f"{r['gain_x1000']:+.1f} [{r['gain_lo']:+.1f}, {r['gain_hi']:+.1f}] | {ph} |")
    md.append(f"\n{res['n_bins']} bins, {res['n_scenes']} scenes, {resC['n_trials']} probe trials. flow_accel boundary AUROC {resB['flow_accel']['auc']:.2f}. "
              f"Pilot 0 rows (L/base) are reference values on the same bins; Holm is over the new signals only.")
    (RESULTS / "events_results_scaled.md").write_text("\n".join(md) + "\n")
    print("\n".join(md))


if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else str(RESULTS / "events_features_scaled.csv"),
         sys.argv[2] if len(sys.argv) > 2 else str(RESULTS / "events_features.csv"),
         sys.argv[3] if len(sys.argv) > 3 else str(RESULTS / "events_models_scaled.json"))


In [ ]:
#@title 4. Smoke test (1 scene x 2 bins per model; downloads the weights)
%cd /content/world-maps
if SMOKE_TEST_FIRST:
    args = " ".join(MODELS)
    !PYTHONPATH=src python scripts/extract_event_signals_scaled.py --models {args} --vlm-id {VLM_ID} --limit 1 --max-bins 2 --out /content/smoke.csv --meta /content/smoke.json 2>&1 | grep -v -i "warn\|Loading weights\|Fetching"
    import pandas as pd, json
    display(pd.read_csv("/content/smoke.csv"))
    print(json.dumps({k: v.get("status") for k, v in json.load(open("/content/smoke.json")).items()}, indent=1))
else:
    print("skipped")

In [ ]:
#@title 5. Full extraction (all scenes; resumable per model if the runtime restarts)
%cd /content/world-maps
args = " ".join(MODELS)
limit = f"--limit {LIMIT_SCENES}" if LIMIT_SCENES else ""
!PYTHONPATH=src python scripts/extract_event_signals_scaled.py --models {args} --vlm-id {VLM_ID} {limit} 2>&1 | grep -v -i "warn\|Loading weights\|Fetching"
import json; print(json.dumps(json.load(open("results/events_models_scaled.json")), indent=1))

In [ ]:
#@title 6. Analysis (CPU, about 2 min): boundary AUROC and held-out human prediction for every signal
%cd /content/world-maps
!PYTHONPATH=src python scripts/analyze_events_scaled.py

In [ ]:
#@title 7. Package results for download
%cd /content/world-maps
!zip -q -j /content/world_maps_scaled_results.zip results/events_features_scaled.csv results/events_models_scaled.json results/events_results_scaled.json results/events_results_scaled.md scripts/extract_event_signals_scaled.py scripts/analyze_events_scaled.py
!ls -la /content/world_maps_scaled_results.zip
from google.colab import files
files.download("/content/world_maps_scaled_results.zip")

### If something fails
- **Out of memory on the VLM:** set `VLM_ID = "Qwen/Qwen3-VL-4B-Instruct"` (or `-2B-`) in cell 1 and rerun cells 5–7. Other models are unaffected; a failed model is recorded in `events_models_scaled.json` and the rest continue.
- **Runtime disconnects:** rerun cells 2, 3 and 5; completed models are skipped (results are in `results/events_features_scaled.csv`). To keep results across sessions, mount Drive and copy `results/` there.
- **DINOv3:** requires accepting the model terms on the Hub and an `HF_TOKEN` secret in Colab.